# 🦀 Day 7: Mini-Project — Testing Framework Macro
---

Today we build a **custom testing framework** using macros!

- **Project overview**: Custom `#[my_test]` with setup/teardown
- **Features**: Setup before each test, teardown after, timing, result reporting
- **macro_rules!** version: `test_suite!` — runnable in EvCxR
- **Assertion helpers**: `assert_approx_eq!`, `assert_contains!`
- **Proc-macro version** design (Cargo project)

In [ ]:
// test_suite! macro — macro_rules! version (runnable in EvCxR)
struct TestResult {
    name: &'static str,
    passed: bool,
    elapsed_ms: u128,
}

macro_rules! test_suite {
    (setup => $setup:block, teardown => $teardown:block, $($name:ident => $body:block),* $(,)?) => {
        {
            let mut results = Vec::new();
            $( {
                let start = std::time::Instant::now();
                $setup
                let passed = std::panic::catch_unwind(|| $body).is_ok();
                $teardown
                let elapsed = start.elapsed().as_millis();
                results.push(TestResult { name: stringify!($name), passed, elapsed_ms: elapsed });
            } )*
            results
        }
    };
}

let results = test_suite!(
    setup => { /* e.g. create temp dir */ },
    teardown => { /* cleanup */ },
    test_add => { assert_eq!(1 + 1, 2); },
    test_sub => { assert_eq!(5 - 3, 2); },
);

for r in &results {
    let status = if r.passed { "PASS" } else { "FAIL" };
    let color = if r.passed { "\x1b[32m" } else { "\x1b[31m" };
    println!("{} {} {} ({}ms)\x1b[0m", color, status, r.name, r.elapsed_ms);
}
println!("\n{} passed, {} failed", results.iter().filter(|r| r.passed).count(), results.iter().filter(|r| !r.passed).count());

In [ ]:
// Assertion helpers: assert_approx_eq!, assert_contains!
macro_rules! assert_approx_eq {
    ($left:expr, $right:expr, $eps:expr) => {
        assert!(
            ($left - $right).abs() < $eps,
            "assertion failed: |{} - {}| < {}",
            $left, $right, $eps
        );
    };
}

macro_rules! assert_contains {
    ($haystack:expr, $needle:expr) => {
        assert!(
            $haystack.contains($needle),
            "assertion failed: {:?} does not contain {:?}",
            $haystack, $needle
        );
    };
}

assert_approx_eq!(0.1 + 0.2, 0.3, 0.0001);
assert_contains!("hello world", "world");
println!("Assertion helpers work!");

In [ ]:
// Proc-macro version design: #[my_test] attribute
//
// #[proc_macro_attribute]
// pub fn my_test(attr: TokenStream, item: TokenStream) -> TokenStream {
//     // attr could be: #[my_test] or #[my_test(should_panic)] or #[my_test(ignore)]
//     // Parse ItemFn, wrap body with setup/teardown, register in test harness
//     // For integration: use inventory crate or linkme for test collection
//     quote! { ... }.into()
// }
//
// User: #[my_test] fn test_foo() { assert_eq!(1, 1); }
//       #[my_test(should_panic)] fn test_panic() { panic!("expected"); }

println!("Proc-macro #[my_test] would need test collection (e.g. inventory crate).");

## 📝 Exercise: Add #[should_panic] or test filtering

Extend the `test_suite!` macro to support:
- **Option A**: `test_panic => #[should_panic] { panic!("expected"); }` — expect panic, pass if it panics
- **Option B**: Filter tests by name pattern, e.g. only run tests matching `test_math_*`

Implement one of these in the macro_rules! version.

In [ ]:
// YOUR CODE HERE
// Extend test_suite! with should_panic support:
// test_panic => should_panic { panic!("expected"); }
// Or add a filter: test_suite!(filter => "math", ...)

## 🎯 Key Takeaways

1. **test_suite!** combines setup, teardown, and test blocks with timing and result collection
2. **Colored output** (green pass, red fail) improves readability
3. **assert_approx_eq!** and **assert_contains!** extend the assertion toolkit
4. **Proc-macro #[my_test]** would use `inventory` or similar for test registration
5. **should_panic** and **filtering** are natural extensions to the framework
6. Macros let you build **DSLs** — custom syntax for your domain

---
**Week 18 Complete! Next week: Unsafe Rust & FFI!**